# 06 · Task 3 execution-guided self-correction (🌐🔑)

Cells marked 🌐 call the OpenRouter API and need `OPENROUTER_API_KEY`. EGSC never sees gold SQL; gold is used only in the evaluation cells.

In [45]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "common").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from common import config
# Optional override of the project root for outputs/results (used for offline fixture tests).
if os.environ.get("A1_ROOT_OVERRIDE"):
    config.ROOT_DIR = Path(os.environ["A1_ROOT_OVERRIDE"])

LIMIT = None  # set to an int for a smoke run; falls back to A1_LIMIT
LIMIT = config.resolve_limit(LIMIT)
OUT = config.out_root(LIMIT)
RES = config.results_root(LIMIT)
print("LIMIT =", LIMIT, "| outputs:", OUT, "| results:", RES)

LIMIT = None | outputs: /home/nguyen/School/CISC_451/assignment1/outputs | results: /home/nguyen/School/CISC_451/assignment1/results


In [46]:
from common import report, data, evaluate
from common.config import RunConfig
from common.egsc import run_egsc
from common.runner import read_predictions

cfg = RunConfig(run_name="qwen25_7b_egsc", model_id="qwen/qwen-2.5-7b-instruct",
                temperature=0.0, max_tokens=512, retry_on_empty=False, limit=LIMIT)
examples = data.load_examples(LIMIT)
orig_dir = OUT / "qwen25_7b"
orig_preds = read_predictions(orig_dir / "preds.txt")
orig_pe = pd.read_csv(orig_dir / "per_example.csv")
assert len(orig_preds) == len(examples), (len(orig_preds), len(examples))
print(cfg.run_name, cfg.fingerprint()[:12], len(examples), "examples")

qwen25_7b_egsc 22cb482da0da 1034 examples


## 🌐 Run EGSC (resumable)

In [39]:
import time
from common.llm_client import OpenRouterClient
client = OpenRouterClient(api_key=config.load_api_key(required=True))
t0 = time.monotonic()
stats = run_egsc(cfg, examples, orig_preds, client)
egsc_wall_s = time.monotonic() - t0
print({k: v for k, v in vars(stats).items() if k != "final_sqls"}, "| wall s:", round(egsc_wall_s, 1))

{'n_examples': 1034, 'n_skipped': 0, 'n_triggered': 49, 'n_llm_calls': 65, 'prompt_tokens': 65569, 'completion_tokens': 3872, 'raw_path': '/home/nguyen/School/CISC_451/assignment1/outputs/qwen25_7b_egsc/raw.jsonl', 'preds_path': '/home/nguyen/School/CISC_451/assignment1/outputs/qwen25_7b_egsc/preds.txt'} | wall s: 43.4


## Evaluate final predictions

In [47]:
def load_egsc(run_dir):
    recs = {}
    with open(run_dir / "raw.jsonl", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    r = json.loads(line); recs.setdefault(int(r["example_id"]), r)
                except (json.JSONDecodeError, KeyError):
                    pass
    return recs

def evaluate_egsc(run_cfg):
    run_dir = run_cfg.output_dir()
    finals = read_predictions(run_dir / "preds.txt")
    s = evaluate.evaluate_predictions(finals, data.gold_lines(LIMIT), None, None, run_dir,
                                      run_name=run_cfg.run_name)
    gold_path = data.write_gold(run_dir / "gold.txt", LIMIT)
    cli = evaluate.run_official_cli(gold_path, run_dir / "preds.txt")
    mism = evaluate.compare_totals(s.table, cli)
    evaluate.record_cli_check(run_dir, mism)
    print("CLI totals check:", "PASS" if not mism else f"FAIL {mism}")
    return s.per_example, load_egsc(run_dir)

egsc_pe, egsc_recs = evaluate_egsc(cfg)

CLI totals check: PASS


## Results

In [48]:
main = report.egsc_main_table(orig_pe, egsc_pe)
report.export_table(main, "task3_egsc_main", "Qwen2.5-7B original vs primary EGSC (retry on execution "
                    "error/timeout, up to 2 retries): EX/EM overall and per difficulty. unparseable = "
                    "predictions the official parser cannot parse (EM=0); em_parseable = EM on the "
                    "parseable subset; ex_unparseable = EX on the unparseable subset.", RES)
display(main)

def summarize(pe_final, recs, label):
    o = orig_pe.sort_values("example_id"); f = pe_final.sort_values("example_id")
    trig = [bool(recs[int(e)]["triggered"]) for e in o["example_id"]]
    counts = report.egsc_counts(o["ex"], f["ex"], trig)
    mc = report.mcnemar_exact(o["ex"], f["ex"])
    pt = sum(a.get("prompt_tokens") or 0 for r in recs.values() for a in r["attempts"][1:])
    ct = sum(a.get("completion_tokens") or 0 for r in recs.values() for a in r["attempts"][1:])
    lat = sum(a.get("latency_s") or 0 for r in recs.values() for a in r["attempts"][1:])
    calls = sum(r["n_llm_calls"] for r in recs.values())
    row = {"system": label, **counts, "extra_llm_calls": calls, "extra_prompt_tokens": pt,
           "extra_completion_tokens": ct, "extra_llm_time_s": lat,
           "extra_cost_usd": report.cost_usd(pt, ct, cfg.model_id),
           "mcnemar_b_fixed": mc["b"], "mcnemar_c_broken": mc["c"], "mcnemar_p": mc["p"]}
    return pd.DataFrame([row])

cnt = summarize(egsc_pe, egsc_recs, "primary EGSC")
report.export_table(cnt, "task3_egsc_counts", "Primary EGSC: triggered/fixed/broken counts, extra "
                    "LLM calls, tokens, time and cost, and exact McNemar test on paired EX. "
                    + report.price_note([cfg.model_id]), RES)
display(cnt)

,system,ex_easy,em_easy,ex_medium,em_medium,ex_hard,em_hard,ex_extra,em_extra,ex_all,em_all,n
0,original,0.870968,0.520161,0.804933,0.282511,0.557471,0.224138,0.518072,0.066265,0.733075,0.294971,1034
1,primary EGSC,0.875000,0.520161,0.809417,0.282511,0.591954,0.235632,0.542169,0.072289,0.745648,0.297872,1034


,system,n,triggered,fixed,broken,unchanged,extra_llm_calls,extra_prompt_tokens,extra_completion_tokens,extra_llm_time_s,extra_cost_usd,mcnemar_b_fixed,mcnemar_c_broken,mcnemar_p
0,primary EGSC,1034,49,13,0,1021,65,65569,3872,39.807372,NaN,13,0,0.000244


## Examples: fixed and broken

In [49]:
def show(kind, n=2):
    o = orig_pe.set_index("example_id")["ex"]; f = egsc_pe.set_index("example_id")["ex"]
    want = (0, 1) if kind == "fixed" else (1, 0)
    ids = [e for e in o.index if (o[e], f[e]) == want][:n]
    exs = {ex.example_id: ex for ex in examples}
    print(f"=== {kind}: {len(ids)} shown ===")
    for e in ids:
        r, ex = egsc_recs[int(e)], exs[int(e)]
        print(f"\n[{e}] {ex.db_id}: {ex.question}\n  original: {r['original_sql']}")
        for a in r["attempts"][1:]:
            print(f"  feedback: {a.get('feedback')}\n  retry {a['k']}: {a['sql']}")
        print(f"  final:    {r['final_sql']}\n  gold:     {ex.gold_sql}")
show("fixed"); show("broken")

=== fixed: 2 shown ===

[103] car_1: Which distinct car models are the produced after 1980?
  original: SELECT DISTINCT T2.Model FROM cars_data AS T1 JOIN model_list AS T2 ON T1.MakeId = T2.ModelId WHERE T1.Year > 1980
  feedback: Execution error: OperationalError: no such column: T1.MakeId
  retry 1: SELECT DISTINCT T3.Model FROM cars_data AS T1 JOIN car_names AS T2 ON T1.Id = T2.MakeId JOIN model_list AS T3 ON T2.Model = T3.Model WHERE T1.Year > 1980
  final:    SELECT DISTINCT T3.Model FROM cars_data AS T1 JOIN car_names AS T2 ON T1.Id = T2.MakeId JOIN model_list AS T3 ON T2.Model = T3.Model WHERE T1.Year > 1980
  gold:     SELECT DISTINCT T1.model FROM MODEL_LIST AS T1 JOIN CAR_NAMES AS T2 ON T1.model  =  T2.model JOIN CARS_DATA AS T3 ON T2.MakeId  =  T3.id WHERE T3.year  >  1980;

[167] car_1: Among the cars with more than lowest horsepower, which ones do not have more than 3 cylinders? List the car makeid and make name.
  original: SELECT cn.MakeId, m.Make FROM cars_data cd JOIN 

## Ablation: retry on empty result (🌐, off by default)

Separate run `qwen25_7b_egsc_empty` with `retry_on_empty=True`. Reported only as an ablation.

In [50]:
RUN_EMPTY_ABLATION = False

In [51]:
if RUN_EMPTY_ABLATION:
    cfg_empty = RunConfig(run_name="qwen25_7b_egsc_empty", model_id="qwen/qwen-2.5-7b-instruct",
                          temperature=0.0, max_tokens=512, retry_on_empty=True, limit=LIMIT)
    run_egsc(cfg_empty, examples, orig_preds, client)
    empty_pe, empty_recs = evaluate_egsc(cfg_empty)
    abl = report.egsc_ablation_table(orig_pe, empty_pe)
    report.export_table(abl, "task3_egsc_ablation_empty", report.ablation_caption(
        "Qwen2.5-7B original vs EGSC variant that also retries zero-row results"), RES)
    acnt = summarize(empty_pe, empty_recs, f"EGSC ({report.ABLATION_LABEL})")
    report.export_table(acnt, "task3_egsc_ablation_empty_counts", report.ablation_caption(
        "EGSC zero-row variant: counts, extra cost and McNemar test"), RES)
    display(abl); display(acnt)
else:
    print("Ablation skipped (RUN_EMPTY_ABLATION=False).")

Ablation skipped (RUN_EMPTY_ABLATION=False).
